In [1]:
import pandas as pd

df = pd.read_csv("../raw/1522_motivo_chiamata.csv")

df.head()

,FREQ,Frequenza,REF_AREA,Territorio,DATA_TYPE,Indicatore,PURPOSE_CALL,Motivi della chiamata,SEX,Sesso,NATIONALITY,Nazionalità,TIME_PERIOD,Osservazione,OBS_STATUS
0,A,Annuale,IT,Italia,USERS,Utenti del 1522,VICT_STALK_SEEK,Richiesta di aiuto vittima di stalking,M,Maschi,WORLD,Mondo,2015.0,101.0,NaN
1,A,Annuale,IT,Italia,USERS,Utenti del 1522,VICT_STALK_SEEK,Richiesta di aiuto vittima di stalking,M,Maschi,WORLD,Mondo,2016.0,136.0,NaN
2,A,Annuale,IT,Italia,USERS,Utenti del 1522,VICT_STALK_SEEK,Richiesta di aiuto vittima di stalking,M,Maschi,WORLD,Mondo,2017.0,109.0,NaN
3,A,Annuale,IT,Italia,USERS,Utenti del 1522,VICT_STALK_SEEK,Richiesta di aiuto vittima di stalking,M,Maschi,WORLD,Mondo,2018.0,68.0,NaN
4,A,Annuale,IT,Italia,USERS,Utenti del 1522,VICT_STALK_SEEK,Richiesta di aiuto vittima di stalking,M,Maschi,WORLD,Mondo,2019.0,65.0,NaN


In [2]:
# Per ogni colonna: numero di valori unici + elenco valori

for col in df.columns:
    print("=" * 60)
    print(f"Colonna: {col}")
    print(f"Numero valori unici: {df[col].nunique(dropna=False)}")
    print("Valori unici:")
    print(df[col].unique())
    print("\n")

Colonna: FREQ
Numero valori unici: 389
Valori unici:
['A'
 'A,Annuale,ITC2,\'Valle d"\'Aosta / Vallée d"\'Aoste\',USERS,Utenti del 1522,VICT_STALK_SEEK,Richiesta di aiuto vittima di stalking,M,Maschi,WORLD,Mondo,2019,1,,'
 'A,Annuale,ITC2,\'Valle d"\'Aosta / Vallée d"\'Aoste\',USERS,Utenti del 1522,VICT_STALK_SEEK,Richiesta di aiuto vittima di stalking,M,Maschi,IT,Italia,2019,1,,'
 'A,Annuale,ITC2,\'Valle d"\'Aosta / Vallée d"\'Aoste\',USERS,Utenti del 1522,VICT_STALK_SEEK,Richiesta di aiuto vittima di stalking,F,Femmine,WORLD,Mondo,2015,1,,'
 'A,Annuale,ITC2,\'Valle d"\'Aosta / Vallée d"\'Aoste\',USERS,Utenti del 1522,VICT_STALK_SEEK,Richiesta di aiuto vittima di stalking,F,Femmine,WORLD,Mondo,2017,2,,'
 'A,Annuale,ITC2,\'Valle d"\'Aosta / Vallée d"\'Aoste\',USERS,Utenti del 1522,VICT_STALK_SEEK,Richiesta di aiuto vittima di stalking,F,Femmine,WORLD,Mondo,2018,2,,'
 'A,Annuale,ITC2,\'Valle d"\'Aosta / Vallée d"\'Aoste\',USERS,Utenti del 1522,VICT_STALK_SEEK,Richiesta di aiuto vittima 

In [3]:
# Rimozione colonne non informative o ridondanti:
# - colonne costanti (un solo valore per tutto il dataset)
# - colonne duplicate sul piano informativo
# - colonne completamente vuote

cols_to_drop = [
    "FREQ",               
    "Frequenza",          
    "REF_AREA",          
    "DATA_TYPE",          
    "Indicatore", 
    "PURPOSE_CALL",  
    "SEX", 
    "NATIONALITY",      
    "OBS_STATUS" 
]

df = df.drop(columns=cols_to_drop)

# Verifica struttura finale
print("Nuova shape:", df.shape)
print("Colonne rimanenti:", df.columns.tolist())


Nuova shape: (23421, 6)
Colonne rimanenti: ['Territorio', 'Motivi della chiamata', 'Sesso', 'Nazionalità', 'TIME_PERIOD', 'Osservazione']


In [4]:
# Standardizzazione nomi colonne:
# - tutto minuscolo
# - nomi più chiari e coerenti

df = df.rename(columns={
    "Territorio": "territorio",
    "Nazionalità": "nazionalità_utente",
    "Motivi della chiamata": "motivo_chiamata",
    "Sesso": "sesso",
    "TIME_PERIOD": "periodo",
    "Osservazione": "numero_vittime"
})


# Eliminazione delle righe completamente vuote 
df = df.dropna(how="all")

# Stampa delle colonne e dei valori unici che contengono
for col in df.columns:
    print("\nColonna:", col)
    print(df[col].unique())


Colonna: territorio
['Italia' 'Nord-ovest' 'Piemonte' 'Liguria' 'Lombardia' 'Nord-est'
 'Trentino Alto Adige / Südtirol' 'Veneto' 'Friuli-Venezia Giulia'
 'Emilia-Romagna' 'Centro' 'Toscana' 'Umbria' 'Marche' 'Lazio' 'Sud'
 'Abruzzo' 'Molise' 'Campania' 'Puglia' 'Basilicata' 'Calabria' 'Isole'
 'Sicilia' 'Sardegna' 'Non indicato']

Colonna: motivo_chiamata
['Richiesta di aiuto vittima di stalking'
 'Richiesta di aiuto vittima di violenza'
 'Richiesta di aiuto per discriminazione'
 'Segnalazione di un caso di violenza' 'Emergenza'
 'Informazioni sul servizio 1522'
 'Informazioni sui Centri Antiviolenza Nazionali'
 'Informazioni giuridiche'
 'Informazioni per professionisti sulle procedure da adottare in caso di violenza'
 'Informazioni sulla responsabilità giuridica degli operatori dei servizi pubblici'
 'Segnalazione disfunzione servizi pubblici/privati'
 'Segnalazione di informazione scorretta sui media'
 'Chiamata internazionale fuori orario'
 'Numeri utili per chiamate fuori target

In [5]:
# Pulizia colonna territorio:
# - uniformazione minuscolo
# - correzione denominazioni incoerenti / verbose

df["territorio"] = (
    df["territorio"]
    .str.lower()
    .str.strip()
    .str.replace("-", " ", regex=False)
)

df["territorio"] = df["territorio"].replace({
    "trentino alto adige / südtirol": "trentino alto adige"
})


# Uniformazione minuscolo per sesso, motivo_chiamata, nazionalità_utente
df["sesso"] = df["sesso"].astype(str).str.lower().str.strip()
df["motivo_chiamata"] = df["motivo_chiamata"].astype(str).str.lower().str.strip()
df["nazionalità_utente"] = df["nazionalità_utente"].astype(str).str.lower().str.strip()


# Nazionalità: mondo + paesi esteri -> esteri, e uniformazione non indicato
df["nazionalità_utente"] = df["nazionalità_utente"].replace({
    "mondo": "estero",
    "paesi esteri": "estero",
    "non indicato": "non indicato"
})


# Rimozione voci aggregate per evitare doppio conteggio
df = df[
    (df["sesso"] != "totale") &
    (df["motivo_chiamata"] != "chiamate valide")
]


# Conversione colonne numeriche (toglie i .0)
df["periodo"] = df["periodo"].astype(int)
df["numero_vittime"] = df["numero_vittime"].astype(int)


# Verifica finale post-pulizia (struttura + qualità base)
display(df.head(10))
df.info()

print("\nMissing values per colonna:")
display(df.isna().sum().sort_values(ascending=False))

print("\nNumero valori unici per colonna:")
display(df.nunique(dropna=False).sort_values(ascending=False))

print("\nRighe duplicate (identiche su tutte le colonne):", df.duplicated().sum())

for col in df.columns:
    print("\nColonna:", col)
    if col == "periodo":
        print(sorted(df[col].astype(int).unique().tolist()))
    else:
        print(df[col].unique())

,territorio,motivo_chiamata,sesso,nazionalità_utente,periodo,numero_vittime
0,italia,richiesta di aiuto vittima di stalking,maschi,estero,2015,101
1,italia,richiesta di aiuto vittima di stalking,maschi,estero,2016,136
2,italia,richiesta di aiuto vittima di stalking,maschi,estero,2017,109
3,italia,richiesta di aiuto vittima di stalking,maschi,estero,2018,68
4,italia,richiesta di aiuto vittima di stalking,maschi,estero,2019,65
5,italia,richiesta di aiuto vittima di stalking,maschi,estero,2020,127
6,italia,richiesta di aiuto vittima di stalking,maschi,estero,2021,145
7,italia,richiesta di aiuto vittima di stalking,maschi,estero,2022,85
8,italia,richiesta di aiuto vittima di stalking,maschi,estero,2023,114
9,italia,richiesta di aiuto vittima di stalking,maschi,estero,2024,144


<class 'pandas.core.frame.DataFrame'>
Index: 13023 entries, 0 to 23294
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   territorio          13023 non-null  object
 1   motivo_chiamata     13023 non-null  object
 2   sesso               13023 non-null  object
 3   nazionalità_utente  13023 non-null  object
 4   periodo             13023 non-null  int64 
 5   numero_vittime      13023 non-null  int64 
dtypes: int64(2), object(4)
memory usage: 712.2+ KB

Missing values per colonna:


territorio            0
motivo_chiamata       0
sesso                 0
nazionalità_utente    0
periodo               0
numero_vittime        0
dtype: int64


Numero valori unici per colonna:


numero_vittime        1045
territorio              26
motivo_chiamata         15
periodo                 10
nazionalità_utente       3
sesso                    3
dtype: int64


Righe duplicate (identiche su tutte le colonne): 86

Colonna: territorio
['italia' 'nord ovest' 'piemonte' 'liguria' 'lombardia' 'nord est'
 'trentino alto adige' 'veneto' 'friuli venezia giulia' 'emilia romagna'
 'centro' 'toscana' 'umbria' 'marche' 'lazio' 'sud' 'abruzzo' 'molise'
 'campania' 'puglia' 'basilicata' 'calabria' 'isole' 'sicilia' 'sardegna'
 'non indicato']

Colonna: motivo_chiamata
['richiesta di aiuto vittima di stalking'
 'richiesta di aiuto vittima di violenza'
 'richiesta di aiuto per discriminazione'
 'segnalazione di un caso di violenza' 'emergenza'
 'informazioni sul servizio 1522'
 'informazioni sui centri antiviolenza nazionali'
 'informazioni giuridiche'
 'informazioni per professionisti sulle procedure da adottare in caso di violenza'
 'informazioni sulla responsabilità giuridica degli operatori dei servizi pubblici'
 'segnalazione disfunzione servizi pubblici/privati'
 'segnalazione di informazione scorretta sui media'
 'chiamata internazionale fuori orario

In [6]:
# Numero righe prima
print("Righe prima:", len(df))

# Rimozione duplicati completi (tutte le colonne)
df = df.drop_duplicates()

# Numero righe dopo
print("Righe dopo:", len(df))

Righe prima: 13023
Righe dopo: 12937


In [7]:
# Salvataggio dataset pulito

output_path = "../data_clean/1522_motivo_chiamata.csv"

df.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("File salvato in:", output_path)

File salvato in: ../data_clean/1522_motivo_chiamata.csv
